# overview
testing the influence of domain list; barplot to compare with the original results

NO k (number of niches) or BASS reuslts included

(deprecated) get batch resutls first then plot the performance


In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from src.utils import *
import src.prompt as prompt
from src.data_loader import load_spatial_data_csv


In [ ]:
from openai import OpenAI
import pandas as pd
import os
import re
import time
import json
client = OpenAI()

# get data from OpenAI
based on test_domain_batchid.txt

In [ ]:
# # Parse the test_domain_batchid.txt file to extract sample_id, test_id, and batch_id combinations
# def parse_batch_file(file_path):
#     """Parse the batch file to extract sample_id, test_id, and batch_id combinations"""
#     combinations = []
    
#     with open(file_path, 'r') as f:
#         lines = [line.strip() for line in f.readlines() if line.strip()]
    
#     i = 0
#     while i < len(lines):
#         line = lines[i]
        
#         # Check if line contains sample_id-test_id pattern
#         if '-' in line and not line.startswith('batch_'):
#             sample_id, test_id = line.split('-')
#             batch_ids = []
            
#             # Collect all batch_ids for this sample_id-test_id combination
#             i += 1
#             while i < len(lines) and lines[i].startswith('batch_'):
#                 batch_ids.append(lines[i])
#                 i += 1
            
#             # Add all combinations for this sample_id-test_id
#             for batch_id in batch_ids:
#                 combinations.append({
#                     'sample_id': sample_id,
#                     'test_id': test_id, 
#                     'batch_id': batch_id
#                 })
#         else:
#             i += 1
    
#     return combinations

# # Parse the batch file
# batch_combinations = parse_batch_file('test_domain_batchid.txt')
# print(f"Found {len(batch_combinations)} combinations to process")

# # Initialize result dataframe
# result_df = pd.DataFrame()

In [ ]:
# # Main processing loop for all combinations
# # Track replicate numbers for each sample_id-test_id combination
# seen_combinations = {}

# for idx, combination in enumerate(batch_combinations):
#     sample_id = combination['sample_id']
#     test_id = combination['test_id'] 
#     batch_id = combination['batch_id']
    
#     # Create a key for the combination
#     combination_key = f"{sample_id}-{test_id}"
    
#     # Update replicate number based on whether we've seen this combination before
#     if combination_key in seen_combinations:
#         seen_combinations[combination_key] += 1
#     else:
#         seen_combinations[combination_key] = 1
    
#     rep = seen_combinations[combination_key]
#     print(f"\\nProcessing combination {idx+1}/{len(batch_combinations)}: {sample_id}-{test_id} with {batch_id}")
    
#     try:
#         # Load configuration for current sample
#         config = load_config("configs/config_zeroshot_starmap.yaml")

#         # Update configuration with current sample data
#         config.data_name = sample_id
#         config.replicate = f"_niche{test_id}rep{rep}" 

#         config.refresh_paths()
#
#         # Load spatial data for current sample
#         data_path = str(dataset_dir("starmap", config.data_name))

#         # Load data using the new function
#         adata = load_spatial_data_csv(
#             data_path=data_path,
#             main_data_file="data.csv",
#             celltype_file="celltype.csv", 
#             pos_file="pos.csv",
#             domain_file="domain.csv",
#             config=config,
#             index_col=0,
#             first_column_names=True
#         )

#         # Prepare neighbor data
#         neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
#             adata, config
#         )

#         # Download and save batch results
#         file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
#         save_name = f"response_{config.data_name}_1_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
#         output_file_name = f"{config.output_path}/{save_name}"
        
#         # Create output directory if it doesn't exist
#         os.makedirs(config.output_path, exist_ok=True)
        
#         # Save the response
#         with open(output_file_name, 'w') as file:
#             file.write(file_response.text)  
#         print(f"The response has been saved to {output_file_name}")

#         # Process GPT results
#         gpt_results_df = pd.DataFrame()
#         n_batch = 1
#         for i in range(1, n_batch+1):
#             save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
#             output_file_name = f"{config.output_path}/{save_name}"
            
#             # Read and process the response file
#             with open(output_file_name, 'r', encoding='utf-8') as file:
#                 for line in file:
#                     try:
#                         # Parse each line as JSON
#                         json_data = json.loads(line.strip())
                        
#                         # Extract custom_id and content
#                         custom_id = json_data['custom_id']
#                         content = json_data['response']['body']['choices'][0]['message']['content']

#                         # Extract outputs - handle both JSON format and text format
#                         extract_dict = extract_json_microenvironment(content)

#                         # Add extracted information to dataframe
#                         gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                            
#                     except json.JSONDecodeError:
#                         print(f"Unable to parse JSON string: {line}")
        
#         gpt_results_df.columns = ['zeroshot_gpt4o_mini']

#         # Join results with adata and refine
#         adata.obs = adata.obs.join(gpt_results_df)
#         adata.obs['zeroshot_gpt4o_mini'] = adata.obs['zeroshot_gpt4o_mini'].fillna("unknown")
        
#         # Refine results
#         adata.obs['zeroshot_gpt4o_mini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_gpt4o_mini'])
        
#         # Create output directory for results if it doesn't exist
#         os.makedirs("./gpt4omini_results", exist_ok=True)
        
#         print(f"saving to ./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

#         adata.obs.to_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

#         # Calculate metrics
#         ari = adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini_refined'])
#         nmi = normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini_refined'])

#         # Gather the results
#         r_df = pd.DataFrame({
#             'data_name': [config.data_name],
#             'niche_test': [test_id],
#             'batch_id': [batch_id],
#             'ARI': [ari],
#             'NMI': [nmi]
#         })
#         result_df = pd.concat([result_df, r_df], ignore_index=True)
        
#         print(f"Completed {sample_id}-{test_id}: ARI={ari:.4f}, NMI={nmi:.4f}")
        
#     except Exception as e:
#         print(f"Error processing {sample_id}-{test_id} with {batch_id}: {str(e)}")
#         # Add failed result to track errors
#         r_df = pd.DataFrame({
#             'data_name': [sample_id],
#             'niche_test': [test_id],
#             'batch_id': [batch_id],
#             'ARI': [np.nan],
#             'NMI': [np.nan]
#         })
#         result_df = pd.concat([result_df, r_df], ignore_index=True)
#         continue

# print(f"\\nProcessing complete! Processed {len(batch_combinations)} combinations.")
# print(f"Results shape: {result_df.shape}")
# print("\\nSummary of results:")
# print(result_df.groupby(['data_name', 'niche_test']).agg({'ARI': ['mean', 'std'], 'NMI': ['mean', 'std']}))

In [ ]:
# # Save final results
# result_df.to_csv('examples/results/test_domain_results.csv', index=False)
# print(f"\\nFinal results saved to test_domain_results.csv")
# print(f"Total combinations processed: {len(result_df)}")



In [ ]:
# !!!!!!!!
# IMPORTANT
# !!!!!!!!

# result_df = pd.read_csv("examples/results/test_domain_list_all_metrics.csv")  # test number of niche
result_df = pd.read_csv("examples/results/test_misnomer_all_metrics.csv")  # test name of niches

In [ ]:
original_results_df = pd.read_csv('examples/results/zeroshot_all_metrics_all_replicates.csv')

# select data_type == 'starmap' and model_type == 'gpt4o_mini'
original_results_df = original_results_df[original_results_df['data_type'] == 'starmap']
original_results_df = original_results_df[original_results_df['model_name'] == 'gpt4o_mini']

original_results_df



# plot


In [ ]:
# # plot specific data_name and replicate
# config = load_config("configs/config_zeroshot_starmap.yaml")
# config.data_name = "BZ9"
# config.replicate = f"_niche1256Wrep1" 

# # Load spatial data for current sample
# data_path = str(dataset_dir("starmap", config.data_name))

# # Load data using the new function
# adata = load_spatial_data_csv(
#     data_path=data_path,
#     main_data_file="data.csv",
#     celltype_file="celltype.csv", 
#     pos_file="pos.csv",
#     domain_file="domain.csv",
#     config=config,
#     index_col=0,
#     first_column_names=True
# )
# #adata.obs = pd.read_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
# adata.obs =pd.read_csv(f"examples/results/test_domain_list/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
# sc.pl.scatter(adata, x="x", y="y", color="zeroshot_gpt4o_mini_refined", title =  f"zeroshot_gpt4o_mini_refined")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline
# set font to Arial
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['pdf.fonttype'] = 42



In [ ]:
def plot_metric_comparison_by_data_name(result_df, original_results_df, metric='NMI', 
                                        n_cols=3, figsize_per_subplot=(5, 5), 
                                        bar_color='skyblue', baseline_color='red',
                                        show_values=True, value_format='.3f', niche_tests='auto'):
    """
    Plot metric changes under different niche test conditions by data name.
    
    Parameters:
    -----------
    result_df : pandas.DataFrame
        DataFrame containing niche test results with columns: 'data_name', 'niche_test_id', metric
    original_results_df : pandas.DataFrame  
        DataFrame containing original results with columns: 'data_name', metric
    metric : str, default 'NMI'
        Name of the metric column to plot (e.g., 'NMI', 'ARI', 'Silhouette', etc.)
    n_cols : int, default 3
        Number of columns in subplot grid
    figsize_per_subplot : tuple, default (5, 5)
        Size of each individual subplot (width, height)
    bar_color : str, default 'skyblue'
        Color for the niche test bars
    baseline_color : str, default 'red'
        Color for the original baseline line and shaded area
    show_values : bool, default True
        Whether to show metric values on top of bars
    value_format : str, default '.3f'
        Format string for displaying values
    niche_tests : str, default 'auto'
        Whether to plot all niche tests or a specific list of niche tests
        
    Returns:
    --------
    fig, axes : matplotlib figure and axes objects
    """
    
    # Data preparation - Calculate mean metric for original results (across replicates)
    original_mean_metric = original_results_df.groupby('data_name')[metric].agg(['mean', 'std']).reset_index()
    original_mean_metric.columns = ['data_name', f'original_{metric}_mean', f'original_{metric}_std']

    print(f"\nOriginal {metric} summary:")
    print(original_mean_metric)

    # Get unique data names that appear in both datasets
    common_data_names = set(original_results_df['data_name'].unique()) & set(result_df['data_name'].unique())
    common_data_names = sorted(list(common_data_names))

    print(f"Creating individual plots for {len(common_data_names)} data names: {common_data_names}")

    # Create subplots for individual data names
    n_data = len(common_data_names)
    n_rows = (n_data + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize_per_subplot[0]*n_cols, figsize_per_subplot[1]*n_rows))
    
    # Handle single subplot cases
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)

    for i, data_name in enumerate(common_data_names):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row, col] if n_rows > 1 or n_cols > 1 else axes[0]
        
        # Get data for this specific data_name
        data_subset = result_df[result_df['data_name'] == data_name].copy()
        original_subset = original_mean_metric[original_mean_metric['data_name'] == data_name]
        
        if len(data_subset) > 0 and len(original_subset) > 0:
            if niche_tests == 'auto':
                # Plot niche_test results
                niche_tests = sorted(data_subset['niche_test_id'].unique())
                # sort niche_tests by the length of the string
                niche_tests = sorted(niche_tests, key=lambda x: len(str(x)))
            else:
                niche_tests = niche_tests

            metric_means = []
            metric_stds = []
            
            for niche_test in niche_tests:
                niche_data = data_subset[data_subset['niche_test_id'] == niche_test][metric]
                metric_means.append(niche_data.mean())
                metric_stds.append(niche_data.std() if len(niche_data) > 1 else 0)
            
            # Plot with error bars
            x_pos = range(len(niche_tests))
            bars = ax.bar(x_pos, metric_means, yerr=metric_stds, capsize=5, alpha=0.7, 
                         color=bar_color, label=f'Niche Test {metric}')
            
            # Add original metric as horizontal line
            original_metric = original_subset[f'original_{metric}_mean'].iloc[0]
            original_std = original_subset[f'original_{metric}_std'].iloc[0]
            ax.axhline(y=original_metric, color=baseline_color, linestyle='--', linewidth=2, 
                      label=f'Original {metric}: {original_metric:{value_format}}±{original_std:{value_format}}')
            
            # Fill area for original metric ± std
            ax.fill_between([-0.5, len(niche_tests)-0.5], 
                           [original_metric-original_std]*2, 
                           [original_metric+original_std]*2, 
                           alpha=0.2, color=baseline_color)
            
            ax.set_title(f'{data_name}', fontsize=12, fontweight='bold')
            ax.set_xlabel('Niche Test ID')
            ax.set_ylabel(f'{metric} Score')
            ax.set_xticks(x_pos)
            ax.set_xticklabels(niche_tests, rotation=45)
            ax.legend(fontsize=8)
            # ax.grid(True, alpha=0.3)
            ax.set_ylim(0.3, 0.9)
            
            # Add value labels on bars
            if show_values:
                for j, (bar, mean_val, std_val) in enumerate(zip(bars, metric_means, metric_stds)):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + std_val + 0.01,
                           f'{mean_val:{value_format}}', ha='center', va='bottom', fontsize=8)
        else:
            ax.text(0.5, 0.5, f'No data for {data_name}', ha='center', va='center', 
                   transform=ax.transAxes, fontsize=10)
            ax.set_title(f'{data_name} (No Data)')

    # Hide empty subplots
    for i in range(n_data, n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        if n_rows > 1 or n_cols > 1:
            axes[row, col].set_visible(False)

    plt.tight_layout()
    plt.suptitle(f'{metric} Changes Under Different Niche Test Conditions by Data Name', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    return fig, axes

# Example usage with NMI (original functionality)
fig, axes = plot_metric_comparison_by_data_name(result_df, original_results_df, metric='NMI')
plt.savefig("figures/NMI_vs_num_niches_by_data_name.pdf")
plt.show()


In [ ]:
# using a specific list of niche tests
fig, axes = plot_metric_comparison_by_data_name(
    result_df, 
    original_results_df, 
    metric='NMI', 
    niche_tests=['6_WM', '5_WM', '23_WM', '1_WM'],
    )
plt.savefig("figures/NMI_vs_num_niches_by_data_name_miss.pdf")
plt.show()


In [ ]:
# Example usage with different metrics

# 1. Plot ARI metric
print("="*50)
print("PLOTTING ARI METRIC")
print("="*50)
fig_ari, axes_ari = plot_metric_comparison_by_data_name(
    result_df, original_results_df, 
    metric='ARI',
    bar_color='lightcoral',
    baseline_color='darkgreen'
)
plt.show()

# 2. Plot with custom styling
print("="*50) 
print("PLOTTING COM WITH CUSTOM STYLING")
print("="*50)
fig_custom, axes_custom = plot_metric_comparison_by_data_name(
    result_df, original_results_df,
    metric='COM',
    n_cols=3,  # Different layout
    figsize_per_subplot=(6, 4),  # Larger subplots
    bar_color='mediumpurple',
    baseline_color='orange',
    value_format='.4f'  # More decimal places
)
plt.show()




In [ ]:
def plot_metric_distribution_across_conditions(result_df, original_results_df, metric='NMI',
                                              figsize=(20, 8), show_sample_size=True,
                                              box_color=None, violin_color=None):
    """
    Create overall distribution plots showing metric changes across all data_names and niche_test conditions.
    
    Parameters:
    -----------
    result_df : pandas.DataFrame
        DataFrame containing niche test results with columns: 'data_name', 'niche_test', metric
    original_results_df : pandas.DataFrame  
        DataFrame containing original results with columns: 'data_name', metric
    metric : str, default 'NMI'
        Name of the metric column to plot (e.g., 'NMI', 'ARI', 'Silhouette', etc.)
    figsize : tuple, default (20, 8)
        Figure size as (width, height)
    show_sample_size : bool, default True
        Whether to show sample size annotations on box plot
    box_color : str, optional
        Color for box plot. If None, uses seaborn default
    violin_color : str, optional
        Color for violin plot. If None, uses seaborn default
        
    Returns:
    --------
    fig, (ax1, ax2) : matplotlib figure and axes objects
    plot_df : pandas.DataFrame
        Combined dataset used for plotting
    """
    
    # Get unique data names that appear in both datasets
    common_data_names = set(original_results_df['data_name'].unique()) & set(result_df['data_name'].unique())
    common_data_names = sorted(list(common_data_names))
    
    # Create a comprehensive dataset for plotting
    plot_data = []

    # Add niche test results
    for _, row in result_df.iterrows():
        plot_data.append({
            'data_name': row['data_name'],
            'niche_test': str(row['niche_test']),
            metric: row[metric],
            'condition': f"Niche_{row['niche_test']}",
            'type': 'Niche Test'
        })

    # Add original results
    for data_name in common_data_names:
        original_data = original_results_df[original_results_df['data_name'] == data_name]
        for _, row in original_data.iterrows():
            plot_data.append({
                'data_name': row['data_name'],
                'niche_test': 'Original',
                metric: row[metric],
                'condition': 'Original',
                'type': 'Original'
            })

    plot_df = pd.DataFrame(plot_data)

    print(f"Combined dataset shape: {plot_df.shape}")
    print(f"Conditions: {sorted(plot_df['condition'].unique())}")

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # Box plot by condition
    box_plot = sns.boxplot(data=plot_df, x='condition', y=metric, ax=ax1, color=box_color)
    ax1.set_title(f'{metric} Distribution Across All Conditions', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Condition')
    ax1.set_ylabel(f'{metric} Score')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)

    # Add sample size annotations
    if show_sample_size:
        for i, condition in enumerate(sorted(plot_df['condition'].unique())):
            count = len(plot_df[plot_df['condition'] == condition])
            ax1.text(i, ax1.get_ylim()[1] * 0.95, f'n={count}', ha='center', va='top', fontsize=10)

    # Violin plot by condition
    violin_plot = sns.violinplot(data=plot_df, x='condition', y=metric, ax=ax2, color=violin_color)
    ax2.set_title(f'{metric} Distribution Density Across All Conditions', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Condition')
    ax2.set_ylabel(f'{metric} Score')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    
    return fig, (ax1, ax2), plot_df

# Example usage with NMI (original functionality)
fig, (ax1, ax2), plot_df = plot_metric_distribution_across_conditions(result_df, original_results_df, metric='NMI')
plt.show()


In [ ]:
# Example usage with different metrics and styling options

# 1. Plot ARI distribution with custom colors
print("="*60)
print("PLOTTING ARI DISTRIBUTION ACROSS CONDITIONS")
print("="*60)
fig_ari, (ax1_ari, ax2_ari), plot_df_ari = plot_metric_distribution_across_conditions(
    result_df, original_results_df, 
    metric='ARI',
    box_color='lightcoral',
    violin_color='lightblue'
)
plt.show()

# 2. Plot NMI with custom figure size and no sample size annotations
print("="*60)
print("PLOTTING NMI WITH CUSTOM STYLING")
print("="*60)
fig_custom, (ax1_custom, ax2_custom), plot_df_custom = plot_metric_distribution_across_conditions(
    result_df, original_results_df,
    metric='NMI',
    figsize=(16, 6),  # Smaller figure
    show_sample_size=False,  # No sample size annotations
    box_color='mediumpurple',
    violin_color='lightgreen'
)
plt.show()

# 3. Statistical summary of the distribution data
print("="*60)
print("STATISTICAL SUMMARY BY CONDITION")
print("="*60)
summary_stats = plot_df.groupby('condition')['NMI'].agg(['count', 'mean', 'std', 'min', 'max']).round(4)
print(summary_stats)

# 4. Quick comparison function to see multiple metrics at once
def quick_metric_comparison(result_df, original_results_df, metrics=['NMI', 'ARI']):
    """Quick function to compare multiple metrics side by side"""
    n_metrics = len(metrics)
    fig, axes = plt.subplots(n_metrics, 2, figsize=(20, 6*n_metrics))
    
    if n_metrics == 1:
        axes = axes.reshape(1, -1)
    
    for i, metric in enumerate(metrics):
        # Get the plot data
        _, (ax1, ax2), _ = plot_metric_distribution_across_conditions(
            result_df, original_results_df, 
            metric=metric,
            figsize=(20, 6)  # This won't be used since we're providing axes
        )
        
        # Copy the plots to our subplot
        # Note: This is a simplified version - in practice you'd want to recreate the plots
        # on the provided axes rather than creating new figures
        print(f"Metric: {metric}")
    
    plt.tight_layout()
    return fig, axes

print("="*60)
print("AVAILABLE FUNCTIONS SUMMARY")
print("="*60)
print("1. plot_metric_comparison_by_data_name() - Individual plots for each data_name")
print("2. plot_metric_distribution_across_conditions() - Overall distribution plots")
print("3. Both functions support any metric column in your DataFrames")
print("4. Customize colors, figure size, and other styling options")


In [ ]:
# 3b. Scatter plot comparing original vs niche test NMI
# Calculate difference between original and niche test NMI
comparison_data = []

for data_name in common_data_names:
    original_nmi = original_mean_nmi[original_mean_nmi['data_name'] == data_name]['original_NMI_mean'].iloc[0]
    niche_data = result_df[result_df['data_name'] == data_name]
    
    for niche_test in niche_data['niche_test'].unique():
        niche_nmi = niche_data[niche_data['niche_test'] == niche_test]['NMI'].mean()
        comparison_data.append({
            'data_name': data_name,
            'niche_test': niche_test,
            'original_NMI': original_nmi,
            'niche_NMI': niche_nmi,
            'NMI_difference': niche_nmi - original_nmi,
            'NMI_ratio': niche_nmi / original_nmi if original_nmi != 0 else np.nan
        })

comparison_df = pd.DataFrame(comparison_data)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Scatter plot: Original vs Niche NMI
colors = plt.cm.Set3(np.linspace(0, 1, len(common_data_names)))
for i, data_name in enumerate(common_data_names):
    data_subset = comparison_df[comparison_df['data_name'] == data_name]
    ax1.scatter(data_subset['original_NMI'], data_subset['niche_NMI'], 
               c=[colors[i]], label=data_name, s=60, alpha=0.7)

# Add diagonal line (y=x)
min_val = min(comparison_df['original_NMI'].min(), comparison_df['niche_NMI'].min())
max_val = max(comparison_df['original_NMI'].max(), comparison_df['niche_NMI'].max())
ax1.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='y=x')
ax1.set_xlabel('Original NMI')
ax1.set_ylabel('Niche Test NMI')
ax1.set_title('Original vs Niche Test NMI')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# NMI difference by niche test
sns.boxplot(data=comparison_df, x='niche_test', y='NMI_difference', ax=ax2)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7)
ax2.set_title('NMI Difference (Niche - Original) by Niche Test')
ax2.set_xlabel('Niche Test')
ax2.set_ylabel('NMI Difference')
ax2.grid(True, alpha=0.3)

# NMI difference by data name
sns.boxplot(data=comparison_df, x='data_name', y='NMI_difference', ax=ax3)
ax3.axhline(y=0, color='red', linestyle='--', alpha=0.7)
ax3.set_title('NMI Difference (Niche - Original) by Data Name')
ax3.set_xlabel('Data Name')
ax3.set_ylabel('NMI Difference')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3)

# NMI ratio distribution
ax4.hist(comparison_df['NMI_ratio'].dropna(), bins=20, alpha=0.7, edgecolor='black')
ax4.axvline(x=1, color='red', linestyle='--', alpha=0.7, label='Ratio = 1 (No change)')
ax4.set_xlabel('NMI Ratio (Niche/Original)')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of NMI Ratios')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
